In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 5.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://spending-chastise-bullion.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://spending-chastise-bullion.ngrok-free.dev


True

In [6]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,

                # 💡 為什麼會回應兩次？
                # 因為你在 messages 陣列（中括號）裡面放了兩個 TextMessage 物件！
                # LINE 允許單次回覆最多發送 5 則訊息，這裡你填了兩個，所以手機就會「同時收到兩則」一樣的訊息。
                messages=[TextMessage(text=event.message.text),# 第一則訊息：鸚鵡重複使用者的話
                            TextMessage(text=event.message.text)]# 第二則訊息：再次重複使用者的話
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ua5551030dc64321324b89c4f0ea9257d","events":[{"type":"message","message":{"type":"text","id":"616277930008117504","quoteToken":"qin2wxp4tvKW_VrKvsDYyDdXCsUH7yAw5p7j-BRnqJJyHvF7j-IacPhlH2f9h2DDQFPCjtSn0b9NLU9PXa2-T3IcS70HZi2d4TGJvTumuK0uzTH0OfMxAlZ6m0eybk8FODlnDQJuKqKaXcaO_ISTEw","markAsReadToken":"pAbytaWD95KICy5kVB-zEPC_-tCPlEzMCSlV0R3JGD8xuorT1U3-RiFDLqfkYgYVFBPc6i-E-1lrVaRo1z29lOa9fZaYnQIBZ5vp1n9gobee_KXNXIZlir1F5VFT9gytViQe2N--4YMjhjD2T9mGftMOcWP6iBFShFPjEb-YI1F2ZOjNeY6vHQ7A7T2VYC__j9twa1XkiTDuEA_dx3hz6A","text":"我是溫佩珊"},"webhookEventId":"01KSWYBYKA4JY5E2BQQ0100BPF","deliveryContext":{"isRedelivery":false},"timestamp":1780161575023,"source":{"type":"user","userId":"Ue65570520db252700cb187ff02c70937"},"replyToken":"552fac2da6ea4de88ed90de0dbbee0d1","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='Ue65570520db252700cb187ff02c70937') timestamp=1780161575023 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KSWYBYKA4JY5E2BQQ01

INFO:werkzeug:127.0.0.1 - - [30/May/2026 17:19:36] "POST / HTTP/1.1" 200 -
